In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [15]:
# Make outputs reproducible
np.random.seed(42)

# Nicer float display in pandas
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Part 1 — Data Cleaning & Preprocessing

Real datasets are almost never clean. Before training any ML model we need to handle things like:

- Missing values (`NaN`)
- Duplicated rows
- Inconsistent text (capitalization, whitespace)
- Wrong data types
- Outliers
- Features on very different scales

We will build three small synthetic datasets, each designed to illustrate a specific problem.

In [16]:
# Dataset A: students with missing values
dataset_a = pd.DataFrame({
    "student_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "age":        [20, 21, np.nan, 22, 23, 20, np.nan, 24],
    "study_hours":[10, 12, 8,  np.nan, 15, 9, 11, 14],
    "score":      [78, 82, 65, 70, np.nan, 72, 80, 88],
})

dataset_a

,student_id,age,study_hours,score
0,1,20.000,10.000,78.000
1,2,21.000,12.000,82.000
2,3,NaN,8.000,65.000
3,4,22.000,NaN,70.000
4,5,23.000,15.000,NaN
5,6,20.000,9.000,72.000
6,7,NaN,11.000,80.000
7,8,24.000,14.000,88.000


In [17]:
# Step 1: detect missing values per column
print("Missing values per column:")
print(dataset_a.isna().sum())

Missing values per column:
student_id     0
age            2
study_hours    1
score          1
dtype: int64


In [18]:
# Step 2: impute (fill) missing values
# Strategy: fill numeric columns with the column mean.
# This is a simple, very common baseline strategy.

dataset_a_clean = dataset_a.copy()

for col in ["age", "study_hours", "score"]:
    mean_value = dataset_a_clean[col].mean()
    dataset_a_clean[col] = dataset_a_clean[col].fillna(mean_value)

dataset_a_clean

,student_id,age,study_hours,score
0,1,20.000,10.000,78.000
1,2,21.000,12.000,82.000
2,3,21.667,8.000,65.000
3,4,22.000,11.286,70.000
4,5,23.000,15.000,76.429
5,6,20.000,9.000,72.000
6,7,21.667,11.000,80.000
7,8,24.000,14.000,88.000


**Note:** The mean is only one of several imputation strategies. Other common options are the **median** (more robust to outliers), the **mode** (useful for categorical columns), or more advanced techniques like KNN imputation. The right choice depends on the data.

In [19]:
# Dataset B: dirty car sales data
dataset_b = pd.DataFrame({
    "brand":  ["Toyota", "toyota ", "HONDA", "Ford", "ford", "Toyota", "Honda ", "TOYOTA"],
    "model":  ["Corolla", "Corolla", "Civic", "Focus", "Focus", "Corolla", "Civic", "Corolla"],
    "price":  ["15000", "15000", "18000", "12000", "12000", "15000", "18500", "15000"],
    "year":   [2018, 2018, 2019, 2017, 2017, 2018, 2020, 2018],
})

dataset_b

,brand,model,price,year
0,Toyota,Corolla,15000,2018
1,toyota,Corolla,15000,2018
2,HONDA,Civic,18000,2019
3,Ford,Focus,12000,2017
4,ford,Focus,12000,2017
5,Toyota,Corolla,15000,2018
6,Honda,Civic,18500,2020
7,TOYOTA,Corolla,15000,2018


In [20]:
# Step 1: check data types.
# Notice that "price" is an 'object' (string), not a number.
dataset_b.dtypes

brand      str
model      str
price      str
year     int64
dtype: object

In [21]:
# Step 2: normalize text in the 'brand' column
# - strip() removes leading/trailing whitespace
# - str.title() converts to Title Case ("toyota" -> "Toyota")
dataset_b_clean = dataset_b.copy()
dataset_b_clean["brand"] = dataset_b_clean["brand"].str.strip().str.title()

In [22]:
# Step 3: convert 'price' from string to numeric
dataset_b_clean["price"] = pd.to_numeric(dataset_b_clean["price"])

In [23]:
# Step 4: remove duplicated rows
print(f"Rows before removing duplicates: {len(dataset_b_clean)}")
dataset_b_clean = dataset_b_clean.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates:  {len(dataset_b_clean)}")

dataset_b_clean

Rows before removing duplicates: 8
Rows after removing duplicates:  4


,brand,model,price,year
0,Toyota,Corolla,15000,2018
1,Honda,Civic,18000,2019
2,Ford,Focus,12000,2017
3,Honda,Civic,18500,2020


In [24]:
# Verify that data types are now correct
dataset_b_clean.dtypes

brand      str
model      str
price    int64
year     int64
dtype: object

### Dataset C — Outliers and Feature Scaling

Outliers are values that are very different from the rest of the data. They can be legitimate, but they can also be errors (e.g., a typo).

Feature scaling is important when features live on very different numerical ranges — for example, a car's weight (thousands of pounds) vs. its number of cylinders (4–8). Without scaling, features with larger ranges can dominate the learning process.

In [ ]:
# Dataset C: car specs with one clear outlier in 'weight'
dataset_c = pd.DataFrame({
    "weight":      [3.50, 3.69, 3.44, 3.43, 4.34, 4.42, 2.37, 25.00],  # last row is a typo
    "horsepower":  [130,  165,  150,  140,  198,  220,  95,   180],
    "cylinders":   [4,    6,    4,    4,    8,    8,    4,    6],
    "mpg":         [18,   15,   18,   16,   15,   14,   24,   17],
})

dataset_c